# Highbay schema/prose fine-tune - v5

Objective: Fine-tune Qwen2.5-1.5B-Instruct for Typed Markdown IR & AST Generation (v5).

In [1]:
# 1. Mount Google Drive & Create Missing Training Directories
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/HighbayGeniusTraining"
TRAINING_OUTPUT_DIR = os.path.join(DRIVE_BASE, "training", "outputs")

os.makedirs(TRAINING_OUTPUT_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
# 2. Setup & Installation
!pip install -q torch transformers datasets unsloth trl

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.7/75.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 136.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3

In [ ]:
# 3. Configuration Parameters
BASE_MODEL   = "unsloth/Qwen2.5-1.5B-Instruct"
RUN_ID       = "v5-qwen2.5-1.5b"
SEED         = 3407
EPOCHS       = 3
LR           = 2e-4
LORA_R       = 16
EVAL_FRACTION = 0.2
TARGET       = "ast"
MAX_SEQ_LENGTH = 2048

In [ ]:
# 4. Model & LoRA Initialization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_R,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

In [ ]:
# 5. Dataset Loading & Formatting
LOCAL_DATASET_PATH = "/content/TrainingExperiments/Highbay/Local/v5/synthetic_typed_markdown_v5.jsonl"
DRIVE_DATASET_PATH = os.path.join(DRIVE_BASE, "datasets", "processed", "synthetic_typed_markdown_v5.jsonl")

if os.path.exists(LOCAL_DATASET_PATH):
    DATASET_PATH = LOCAL_DATASET_PATH
    print(f"Reading dataset from local repository: {DATASET_PATH}")
elif os.path.exists(DRIVE_DATASET_PATH):
    DATASET_PATH = DRIVE_DATASET_PATH
    print(f"Reading dataset from Google Drive: {DATASET_PATH}")
else:
    raise FileNotFoundError(f"Could not find dataset at local path '{LOCAL_DATASET_PATH}' or Drive path '{DRIVE_DATASET_PATH}'")

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.train_test_split(test_size=EVAL_FRACTION, seed=SEED)

PROMPT_TEMPLATE = """Below is an instruction that describes a UI design action or natural language prompt. Write the corresponding Typed Markdown Intermediate Representation (IR).

### Instruction:
{prompt}

### Typed Markdown IR:
{target_ir}"""

def formatting_prompts_func(examples):
    prompts = examples["prompt"]
    targets = examples["target_ir"]
    texts = []
    for p, t in zip(prompts, targets):
        text = PROMPT_TEMPLATE.format(prompt=p, target_ir=t) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# 6. Training Arguments & Trainer Execution
output_run_dir = os.path.join(TRAINING_OUTPUT_DIR, f"outputs_{RUN_ID}")
os.makedirs(output_run_dir, exist_ok=True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["test"],
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = EPOCHS,
        learning_rate = LR,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = output_run_dir,
        seed = SEED,
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 7. Save Adapter directly to Google Drive
adapter_save_path = os.path.join(TRAINING_OUTPUT_DIR, f"adapter_{RUN_ID}")
model.save_pretrained_merged(adapter_save_path, tokenizer, save_method="merged_16bit")
print(f"Adapter successfully saved to {adapter_save_path}")